In [36]:
import xarray as xr
import numpy as np
import glob
import os
import pandas as pd
from pathlib import Path

import sys
sys.path.append("/home/565/dh4185/mn51-dh4185/repos_collab/nesp_bff/")
from utils import locations, model_dict


In [37]:
# -----------------------------------------------------------------------------
# User inputs
# -----------------------------------------------------------------------------

include_boolean_fields = False

file_dir = "/g/data/eg3/nesp_bff/step3_calc_missing_vars/"
historical_dir = "/g/data/eg3/spr548/projects/nesp_bff/data/raw_data/NatHERS/historical/"

scenarios = ["ssp126", "ssp370"]

time_periods = {
    "2030": "2021-2040",
    "2050": "2041-2060",
    "2070": "2061-2080",
}

temp_var = "tas"
temp_metric_name = "annual_mean_temperature"

out_csv = "/home/565/dh4185/mn51-dh4185/repos_collab/nesp_bff/model_temperature_summary_step3output.csv"


locations = list(locations.keys())


In [38]:
# Helper functions
# -----------------------------------------------------------------------------
def extract_gcm_from_filename(filepath):
    """
    Example filename:
    Melbourne_AUS-15_ACCESS-CM2_ssp126_r4i1p1f1_BOM_BARPA-R_v1-r1_1hr_2021-2040_QDC-NatHERS_NatHERSvars.nc
    """
    name = Path(filepath).name
    parts = name.replace(".nc", "").split("_")
    return parts[2]


def maybe_convert_kelvin_to_celsius(da):
    """
    Convert to degC if values strongly suggest Kelvin.
    """
    try:
        sample_mean = float(da.mean(skipna=True).compute())
        if sample_mean > 100:
            return da - 273.15
    except Exception:
        pass
    return da


def reduce_to_gcm_mean(da):
    """
    Mean over time and any remaining non-gcm dimensions.
    """
    da_mean = da.mean(dim="time", skipna=True)
    extra_dims = [d for d in da_mean.dims if d != "gcm"]
    if extra_dims:
        da_mean = da_mean.mean(dim=extra_dims, skipna=True)
    return da_mean


def reduce_to_scalar_mean(da):
    """
    Mean over all dimensions to a scalar.
    """
    dims = list(da.dims)
    if dims:
        da = da.mean(dim=dims, skipna=True)
    return float(da.compute())


def get_historical_mean_temp(loc, historical_dir, temp_var="tas"):
    """
    Read one historical NatHERS file for a location and return annual mean tas.
    Expected filename pattern, e.g.:
    Melbourne_CZ0607_TU_NatHERS_UTC_19891231-20151231.nc
    """
    hist_files = sorted(glob.glob(f"{historical_dir}{loc}_*.nc"))

    if len(hist_files) == 0:
        raise FileNotFoundError(f"No historical file found for location: {loc}")

    if len(hist_files) > 1:
        print(f"Warning: multiple historical files found for {loc}. Using: {Path(hist_files[0]).name}")

    hist_file = hist_files[0]
    ds_hist = xr.open_dataset(hist_file).sel(time=slice("1995","2014"))

    if temp_var not in ds_hist:
        raise KeyError(
            f"'{temp_var}' not found in historical file for {loc}. "
            f"Available variables: {list(ds_hist.data_vars)}"
        )

    da_hist = maybe_convert_kelvin_to_celsius(ds_hist[temp_var])
    hist_mean = reduce_to_scalar_mean(da_hist)

    ds_hist.close()
    return hist_mean


def summarise_one_combination(
    loc,
    ssp,
    future_period,
    time_period,
    file_dir,
    historical_mean_temp,
    temp_var="tas"
):
    pattern = f"{file_dir}{loc}*{ssp}*{time_period}*.nc"
    files = sorted(glob.glob(pattern))

    if len(files) == 0:
        print(f"No files found for {loc}, {ssp}, {future_period}")
        return None

    datasets = []
    for f in files:
        gcm = extract_gcm_from_filename(f)
        ds = xr.open_dataset(f)
        ds = ds.expand_dims(gcm=[gcm])
        datasets.append(ds)

    ds_all = xr.concat(datasets, dim="gcm")

    if temp_var not in ds_all:
        raise KeyError(
            f"'{temp_var}' not found for {loc}, {ssp}, {future_period}. "
            f"Available variables: {list(ds_all.data_vars)}"
        )

    da_temp = maybe_convert_kelvin_to_celsius(ds_all[temp_var])

    annual_mean_temp = reduce_to_gcm_mean(da_temp)
    temp_series = annual_mean_temp.to_series().sort_values()
    temp_series.name = "temp_metric_value"

    n = len(temp_series)
    if n == 0:
        print(f"No valid temperature values for {loc}, {ssp}, {future_period}")
        ds_all.close()
        return None

    ranks = np.arange(1, n + 1)
    median_rank = (n + 1) // 2

    df = pd.DataFrame({
        "location": loc,
        "scenario": ssp,
        "future_period": future_period,
        "gcm": temp_series.index,
        "temp_metric_name": temp_metric_name,
        "temp_metric_value": temp_series.values,
        "historical_temp_metric_value": historical_mean_temp,
        "temp_rank": ranks,
    })

    df["temp_deviation_from_historical"] = (
        df["temp_metric_value"] - df["historical_temp_metric_value"]
    )

    df["is_hottest"] = df["temp_rank"] == n
    df["is_coolest"] = df["temp_rank"] == 1
    df["is_median"] = df["temp_rank"] == median_rank

    df["temp_class"] = ""
    df.loc[df["is_coolest"], "temp_class"] = "coolest"
    df.loc[df["is_median"], "temp_class"] = "median"
    df.loc[df["is_hottest"], "temp_class"] = "hottest"

    df["temp_metric_value"] = df["temp_metric_value"].round(3)
    df["historical_temp_metric_value"] = df["historical_temp_metric_value"].round(3)
    df["temp_deviation_from_historical"] = df["temp_deviation_from_historical"].round(3)

    ds_all.close()
    for ds in datasets:
        ds.close()

    return df


In [41]:
# -----------------------------------------------------------------------------
# Pre-calculate historical annual mean temperature for each location
# -----------------------------------------------------------------------------
historical_means = {}

for loc in locations:
    print(f"Reading historical file for {loc}")
    historical_means[loc] = get_historical_mean_temp(
        loc=loc,
        historical_dir=historical_dir,
        temp_var=temp_var,
    )

historical_df = pd.DataFrame({
    "location": list(historical_means.keys()),
    "historical_temp_metric_value": list(historical_means.values())
}).sort_values("location").reset_index(drop=True)

display(historical_df)

# -----------------------------------------------------------------------------
# Loop over all future combinations
# -----------------------------------------------------------------------------
all_dfs = []

for loc in locations:
    for ssp in scenarios:
        for future_period, time_period in time_periods.items():
            print(f"Processing: {loc}, {ssp}, {future_period}")
            df_one = summarise_one_combination(
                loc=loc,
                ssp=ssp,
                future_period=future_period,
                time_period=time_period,
                file_dir=file_dir,
                historical_mean_temp=historical_means[loc],
                temp_var=temp_var,
            )
            if df_one is not None:
                all_dfs.append(df_one)

# -----------------------------------------------------------------------------
# Concatenate into master table
# -----------------------------------------------------------------------------
if len(all_dfs) == 0:
    raise ValueError("No summary tables were created. Check file patterns and inputs.")

df_master = pd.concat(all_dfs, ignore_index=True)

df_master = df_master.sort_values(
    by=["location", "scenario", "future_period", "temp_rank"],
    ascending=[True, True, True, True]
).reset_index(drop=True)

display(df_master)

# -----------------------------------------------------------------------------
# Write CSV
# -----------------------------------------------------------------------------

if include_boolean_fields:
    df_export = df_master.copy()
else:
    df_export = df_master.drop(columns=["is_hottest", "is_coolest", "is_median"])

    column_order = [
        "location",
        "scenario",
        "future_period",
        "gcm",
        "temp_metric_name",
        "temp_metric_value",
        "historical_temp_metric_value",
        "temp_deviation_from_historical",
        "temp_rank",
        "temp_class",
    ]

    df_export = df_export[column_order]

df_export.to_csv(out_csv, index=False)
print(f"\nSaved summary table to: {out_csv}")

Reading historical file for Melbourne
Reading historical file for Canberra
Reading historical file for Darwin
Reading historical file for Cairns
Reading historical file for Brisbane
Reading historical file for Longreach
Reading historical file for Mildura
Reading historical file for Adelaide
Reading historical file for Perth
Reading historical file for Sydney
Reading historical file for Hobart


,location,historical_temp_metric_value
0,Adelaide,17.103124
1,Brisbane,20.244497
2,Cairns,24.577445
3,Canberra,13.290054
4,Darwin,27.138183
5,Hobart,12.650688
6,Longreach,23.616766
7,Melbourne,14.352122
8,Mildura,17.291055
9,Perth,18.369278


Processing: Melbourne, ssp126, 2030
Processing: Melbourne, ssp126, 2050
Processing: Melbourne, ssp126, 2070
Processing: Melbourne, ssp370, 2030
Processing: Melbourne, ssp370, 2050
Processing: Melbourne, ssp370, 2070
Processing: Canberra, ssp126, 2030
Processing: Canberra, ssp126, 2050
Processing: Canberra, ssp126, 2070
Processing: Canberra, ssp370, 2030
Processing: Canberra, ssp370, 2050
Processing: Canberra, ssp370, 2070
Processing: Darwin, ssp126, 2030
Processing: Darwin, ssp126, 2050
Processing: Darwin, ssp126, 2070
Processing: Darwin, ssp370, 2030
Processing: Darwin, ssp370, 2050
Processing: Darwin, ssp370, 2070
Processing: Cairns, ssp126, 2030
Processing: Cairns, ssp126, 2050
Processing: Cairns, ssp126, 2070
Processing: Cairns, ssp370, 2030
Processing: Cairns, ssp370, 2050
Processing: Cairns, ssp370, 2070
Processing: Brisbane, ssp126, 2030
Processing: Brisbane, ssp126, 2050
Processing: Brisbane, ssp126, 2070
Processing: Brisbane, ssp370, 2030
Processing: Brisbane, ssp370, 2050
Pro

,location,scenario,future_period,gcm,temp_metric_name,temp_metric_value,historical_temp_metric_value,temp_rank,temp_deviation_from_historical,is_hottest,is_coolest,is_median,temp_class
0,Adelaide,ssp126,2030,NorESM2-MM,annual_mean_temperature,17.063000,17.103,1,-0.040,False,True,False,coolest
1,Adelaide,ssp126,2030,CMCC-ESM2,annual_mean_temperature,17.535999,17.103,2,0.433,False,False,False,
2,Adelaide,ssp126,2030,EC-Earth3,annual_mean_temperature,17.620001,17.103,3,0.517,False,False,False,
3,Adelaide,ssp126,2030,MPI-ESM1-2-HR,annual_mean_temperature,17.639000,17.103,4,0.536,False,False,True,median
4,Adelaide,ssp126,2030,ACCESS-CM2,annual_mean_temperature,17.693001,17.103,5,0.589,False,False,False,
...,...,...,...,...,...,...,...,...,...,...,...,...,...
457,Sydney,ssp370,2070,CMCC-ESM2,annual_mean_temperature,20.629000,18.327,3,2.302,False,False,False,
458,Sydney,ssp370,2070,ACCESS-ESM1-5,annual_mean_temperature,20.664000,18.327,4,2.337,False,False,True,median
459,Sydney,ssp370,2070,EC-Earth3,annual_mean_temperature,20.808001,18.327,5,2.481,False,False,False,
460,Sydney,ssp370,2070,ACCESS-CM2,annual_mean_temperature,21.320999,18.327,6,2.994,False,False,False,



Saved summary table to: /home/565/dh4185/mn51-dh4185/repos_collab/nesp_bff/model_temperature_summary_step3output.csv
